In [1]:
# Basic TicTacToe (AI made TicTacToe class for me, I'm lazy haha)
class TicTacToe:
    def __init__(self):
        self.reset()

    def reset(self):
        self.board = [0] * 9
        self.player = 1  # X = 1, O = -1

    def legal_moves(self):
        return [i for i in range(9) if self.board[i] == 0]
        
    def step(self, move):
        if move not in self.legal_moves():
            raise ValueError("Invalid move")

        self.board[move] = self.player
        wins = [
            (0,1,2), (3,4,5), (6,7,8),
            (0,3,6), (1,4,7), (2,5,8),
            (0,4,8), (1,4,7), (2,4,6),
            (0,4,8), (2,4,6)
        ]
        won = any(
            all(self.board[i] == self.player for i in line)
            for line in wins
        )
        done = won or len(self.legal_moves()) == 0
        reward = 1 if won else 0
        if not done:
            self.player *= -1
        return reward, done

    def show(self):
        for i in range(0, 9, 3):
            print(self.board[i:i+3])
        print()

In [7]:
game = TicTacToe()

game.step(0)
game.step(4)
game.step(1)
game.step(5)
reward, state = game.step(2)
print(reward, state)
game.show()

1 True
[1, 1, 1]
[0, -1, -1]
[0, 0, 0]



In [233]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random

In [312]:
class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.seq = nn.Sequential(
        nn.Linear(9, 64), nn.ReLU(),
        nn.Linear(64, 64), nn.ReLU(),
        nn.Linear(64, 9)
        )

    def forward(self, board):
        # when converting board to tensor its shape become (9,)
        x = self.seq(board)
        return x

model = QNetwork()

In [287]:
def choose_action(model, board, epsilon):
    legal_moves = [i for i in range(9) if board[i] == 0]
    if random.random() < epsilon:
        move = random.choice(legal_moves)
    else:
        with torch.no_grad():
            q_values = model(board)
            q_values = q_values.masked_fill(board != 0, float('-inf'))
            move = torch.argmax(q_values).item()
    return move

def get_state(game):
    board = torch.tensor(game.board, dtype=torch.float32) * game.player
    return board

In [288]:
game = TicTacToe()
game.step(0)
get_state(game)

tensor([-1., -0., -0., -0., -0., -0., -0., -0., -0.])

In [290]:
choose_action(model, board, 0)

4

In [337]:
from collections import deque

replay_buffer = deque(maxlen=10_000)

In [352]:
def play_episode(model, epsilon):
    game = TicTacToe()
    done = False
    while not done:
        state = get_state(game)
        move = choose_action(model, state, epsilon)
        reward, done = game.step(move)
        next_state = get_state(game) if not done else torch.zeros(9)
        replay_buffer.append((state, move, reward, next_state, done))
    return game

In [347]:
play_episode(model, 0.5)

In [429]:
batch_size = 32

In [467]:
gamma = 0.99
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
def train_step(model, replay_buffer, batch_size=32):
    if len(replay_buffer) < batch_size:
        return

    batch = random.sample(replay_buffer, batch_size)
    states, actions, rewards, next_states, dones = zip(*batch)

    states = torch.stack(states)
    actions = torch.tensor(actions, dtype=torch.long)
    rewards = torch.tensor(rewards, dtype=torch.float32)
    next_states = torch.stack(next_states)
    dones = torch.tensor(dones, dtype=torch.bool)

    q_values = model(states)
    q_values_selected = q_values.gather(dim=-1, index=actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        q_values_next = model(next_states).masked_fill(next_states != 0, float('-inf'))
        q_values_next_max = q_values_next.max(dim=1).values
        q_targets = rewards - gamma * (1 - dones.float()) * q_values_next_max

    loss = F.mse_loss(q_values_selected, q_targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [468]:
episodes = 20_000

for episode in range(episodes):
    epsilon = max(0.08, 1.0 - episode / 18000)

    play_episode(model, epsilon)
    train_step(model, replay_buffer)

    if (episode + 1) % 1000 == 0:
        print(f"Episode {episode + 1}, epsilon={epsilon:.3f}")

Episode 1000, epsilon=0.945
Episode 2000, epsilon=0.889
Episode 3000, epsilon=0.833
Episode 4000, epsilon=0.778
Episode 5000, epsilon=0.722
Episode 6000, epsilon=0.667
Episode 7000, epsilon=0.611
Episode 8000, epsilon=0.556
Episode 9000, epsilon=0.500
Episode 10000, epsilon=0.445
Episode 11000, epsilon=0.389
Episode 12000, epsilon=0.333
Episode 13000, epsilon=0.278
Episode 14000, epsilon=0.222
Episode 15000, epsilon=0.167
Episode 16000, epsilon=0.111
Episode 17000, epsilon=0.080
Episode 18000, epsilon=0.080
Episode 19000, epsilon=0.080
Episode 20000, epsilon=0.080


In [469]:
def test_all_positions(model):
    model.eval()

    visited = set()
    mistakes = []
    total = 0
    optimal = 0

    def explore(board, player):
        nonlocal total, optimal

        key = (tuple(board), player)
        if key in visited:
            return
        visited.add(key)

        # Skip completed games
        if check_winner(board) != 0 or 0 not in board:
            return

        game = TicTacToe()
        game.board = list(board)
        game.player = player

        # DQN's preferred move
        state = get_state(game)
        dqn_move = choose_action(model, state, epsilon=0)

        # Exact minimax value of the position
        best_value = minimax(tuple(board), player)

        # Value of the action chosen by DQN
        next_board = list(board)
        next_board[dqn_move] = player

        dqn_value = -minimax(tuple(next_board), -player)

        total += 1

        if dqn_value == best_value:
            optimal += 1
        else:
            mistakes.append({
                "board": list(board),
                "player": player,
                "dqn_move": dqn_move,
                "dqn_value": dqn_value,
                "best_value": best_value
            })

        # Explore ALL possible moves,
        # not just DQN's chosen move
        for move in game.legal_moves():
            new_board = list(board)
            new_board[move] = player
            explore(new_board, -player)

    explore([0] * 9, 1)

    print(f"Positions checked: {total}")
    print(f"Optimal moves: {optimal}")
    print(f"Mistakes: {len(mistakes)}")
    print(f"Optimality: {optimal / total * 100:.2f}%")

    return mistakes


mistakes = test_all_positions(model)


Positions checked: 4520
Optimal moves: 4437
Mistakes: 83
Optimality: 98.16%


In [470]:
from collections import Counter

print(Counter(
    (m["best_value"], m["dqn_value"])
    for m in mistakes
))

Counter({(1, 0): 53, (0, -1): 30})
